# Task 3.1 | Q3 KPI snapshot

Run notebook 00 first to refresh the cleaned CSVs and deposit audit. Use a Python kernel with pandas, SQLAlchemy, and JupySQL.

## Q3 KPI summary

> **KPI snapshot (Q3, Jul–Sep):** signups, activation rate, monthly active users, deposit volume, loans disbursed, loan default rate (DPD30), Plus subscribers. Is the 59M deposits headline right? If not, what is, and why the gap?

**Q3 deposits were EGP 19,893,500.** The ~EGP 59M headline matches the raw successful deposit total of EGP 59,477,600 in UTC. Removing exact copies reduced it by EGP 991,800. Keeping valid non-QA users reduced it by another EGP 38,590,000. Applying the Cairo reporting window reduced it by EGP 2,300. The cleaned figure includes EGP 10,297,000 across 89 deposits on September 3.

| KPI | Result | Definition |
|---|---:|---|
| Signups | 6,191 | Cleaned users with a Q3 Cairo signup |
| Activation | 2,357 / 6,191 (38.07%) | KYC verified by extract and a successful deposit within seven days of signup |
| Monthly active users | Jul 702, Aug 1,391, Sep 2,048 | Unique users with a successful transaction of any type in the Cairo month |
| Deposit volume | EGP 19,893,500 | Successful, deduplicated deposits from valid non-QA users |
| Loans disbursed | 1,310 (EGP 1,715,500 principal) | Q3 disbursals |
| Mature DPD30 | 76 / 1,040 (7.31%) | Loans still short at due date plus 30 days among those that reached that date by Nov 5. Another 270 were immature. |
| Lucky Plus subscribers | 93 | Unique subscribers with a subscription overlapping Q3. End date is the first inactive day. |

**Board reporting.** Use the corrected headline and flag the September 3 concentration for reconciliation. MAU rose each month. The remaining KPIs are quarter-level totals or rates, so their monthly trend is unknown.

**Data to request.** Bank or processor settlement records and reversals would help validate the September 3 deposits. Paid invoices, refunds and renewals would support recognized revenue and paid-retention reporting.


In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

ROOT = next(folder for folder in (Path.cwd(), Path.cwd().parent) if (folder / "data").is_dir())
CLEANED = ROOT / "data" / "cleaned"
Q3_START = "2026-07-01"
Q3_END = "2026-10-01"
EXTRACT_DATE = "2026-11-05"
CAIRO_OFFSET = pd.Timedelta(hours=2)

In [2]:
# Load cleaned data into pandas DataFrames 
users = pd.read_csv(CLEANED / "users.csv", parse_dates=["signup_ts", "kyc_verified_ts"])
transactions = pd.read_csv(CLEANED / "transactions.csv", parse_dates=["txn_ts"])
loans = pd.read_csv(CLEANED / "loans.csv")
plus = pd.read_csv(CLEANED / "plus_subscriptions.csv")
deposit_audit = pd.read_csv(CLEANED / "q3_deposit_audit.csv")
print("Read cleaned data into pandas DataFrames")

# Add Converted Cairo Time column 
users["signup_cairo"] = users["signup_ts"] + CAIRO_OFFSET
transactions["txn_cairo"] = transactions["txn_ts"] + CAIRO_OFFSET
report_dates = pd.DataFrame({
    "q3_start": [Q3_START],
    "q3_end": [Q3_END],
    "extract_date": [EXTRACT_DATE],
})
print("UTC +2 Cairo time column added to users and transactions tables")

engine = create_engine("sqlite://")
users.to_sql("users", engine, index=False)
transactions.to_sql("transactions", engine, index=False)
loans.to_sql("loans", engine, index=False)
plus.to_sql("plus", engine, index=False)
deposit_audit.to_sql("deposit_audit", engine, index=False)
report_dates.to_sql("report_dates", engine, index=False)

print(f"Data loaded into in-memory SQLite database")

Read cleaned data into pandas DataFrames
UTC +2 Cairo time column added to users and transactions tables
Data loaded into in-memory SQLite database


In [3]:
%load_ext sql
%sql engine

Tip: You may define configurations in C:\Users\Mohimen\Desktop\lucky_case_study\pyproject.toml or C:\Users\Mohimen\.jupysql\config.

Did not find user configurations in C:\Users\Mohimen\Desktop\lucky_case_study\pyproject.toml.

In [4]:
%%sql
-- Signups: one clean user row per Q3 Cairo signup.

SELECT COUNT(*) AS q3_signups
FROM users 
CROSS JOIN report_dates
WHERE signup_cairo >= q3_start AND signup_cairo < q3_end;

Out[0]: 
+------------+
| q3_signups |
+------------+
|    6191    |
+------------+


Running query in 'sqlite://'

In [5]:
%%sql
-- Check deposits at or after signup, up to and including seven days later.
-- EXISTS counts each verified signup once even when several deposits qualify.

WITH signup_cohort AS (
    SELECT users.*, extract_date
    FROM users CROSS JOIN report_dates
    WHERE signup_cairo >= q3_start AND signup_cairo < q3_end
),
activation AS (
    SELECT user_id,
           CASE WHEN kyc_verified_ts < DATE(extract_date, '+1 day')
                AND EXISTS (
                    SELECT 1
                    FROM transactions AS t
                    WHERE t.user_id = signup_cohort.user_id
                      AND t.txn_type = 'deposit'
                      AND t.status = 'success'
                      AND t.txn_ts >= signup_cohort.signup_ts
                      AND STRFTIME('%Y-%m-%d %H:%M:%f', t.txn_ts)
                          <= STRFTIME('%Y-%m-%d %H:%M:%f', signup_cohort.signup_ts, '+7 days')
                ) THEN 1 ELSE 0 END AS is_activated
    FROM signup_cohort
)
SELECT COUNT(*) AS signups,
       SUM(is_activated) AS activated_users,
       ROUND(100.0 * SUM(is_activated) / COUNT(*), 2) AS activation_rate_pct
FROM activation;

Out[0]: 
+---------+-----------------+---------------------+
| signups | activated_users | activation_rate_pct |
+---------+-----------------+---------------------+
|   6191  |       2357      |        38.07        |
+---------+-----------------+---------------------+


Running query in 'sqlite://'

In [6]:
%%sql
-- MAU: distinct users with a successful transaction of any type in the month.

SELECT STRFTIME('%Y-%m', txn_cairo) AS month,
       COUNT(DISTINCT user_id) AS monthly_active_users
FROM transactions CROSS JOIN report_dates
WHERE status = 'success'
  AND txn_cairo >= q3_start AND txn_cairo < q3_end
GROUP BY month
ORDER BY month;

Out[0]: 
+---------+----------------------+
|  month  | monthly_active_users |
+---------+----------------------+
| 2026-07 |         702          |
| 2026-08 |         1391         |
| 2026-09 |         2048         |
+---------+----------------------+


Running query in 'sqlite://'

In [7]:
%%sql
-- Notebook 00 supplies the raw UTC comparison.
WITH clean_q3_deposits AS (
    SELECT transactions.*
    FROM transactions CROSS JOIN report_dates
    WHERE txn_type = 'deposit' AND status = 'success'
      AND txn_cairo >= q3_start AND txn_cairo < q3_end
),
bridge AS (
    SELECT step, scope, deposits, volume_egp FROM deposit_audit
    UNION ALL
    SELECT 5, 'Apply Cairo calendar', COUNT(*), SUM(amount_egp) FROM clean_q3_deposits
    UNION ALL
    SELECT 6, 'September 3 concentration', COUNT(*), SUM(amount_egp)
    FROM clean_q3_deposits WHERE DATE(txn_cairo) = '2026-09-03'
)
SELECT scope, deposits, volume_egp,
       CASE WHEN step < 6 THEN volume_egp - LAG(volume_egp) OVER (ORDER BY step) END AS change_egp
FROM bridge
ORDER BY step;

Out[0]: 
+---------------------------+----------+------------+-------------+
|           scope           | deposits | volume_egp |  change_egp |
+---------------------------+----------+------------+-------------+
|   Raw UTC, all statuses   |   9131   | 61044000.0 |     None    |
|    Raw UTC, successful    |   7782   | 59477600.0 |  -1566400.0 |
|    Remove exact copies    |   7671   | 58485800.0 |  -991800.0  |
|  Keep valid non-QA users  |   7001   | 19895800.0 | -38590000.0 |
|    Apply Cairo calendar   |   6995   | 19893500.0 |   -2300.0   |
| September 3 concentration |    89    | 10297000.0 |     None    |
+---------------------------+----------+------------+-------------+


Running query in 'sqlite://'

In [8]:
%%sql
-- Disbursements: count loans and sum principal across the Q3 disbursal cohort.

SELECT COUNT(*) AS loans_disbursed, SUM(principal_egp) AS principal_disbursed_egp
FROM loans CROSS JOIN report_dates
WHERE disbursed_date >= q3_start AND disbursed_date < q3_end;

Out[0]: 
+-----------------+-------------------------+
| loans_disbursed | principal_disbursed_egp |
+-----------------+-------------------------+
|       1310      |         1715500         |
+-----------------+-------------------------+


Running query in 'sqlite://'

In [9]:
%%sql
-- Use the supplied date fields. Q3 selects disbursals and November 5 sets maturity.

WITH q3_loans AS (
    SELECT loans.*, extract_date, DATE(due_date, '+30 days') AS dpd30_date,
           principal_egp + fee_egp AS total_due_egp
    FROM loans CROSS JOIN report_dates
    WHERE disbursed_date >= q3_start AND disbursed_date < q3_end
),
loan_flags AS (
    SELECT *,
           amount_repaid_egp >= total_due_egp - 0.01 AS fully_paid,
           last_repayment_date > dpd30_date AS last_payment_after_dpd30,
           dpd30_date <= extract_date AS matured
    FROM q3_loans
),
loan_outcomes AS (
    SELECT loan_id, matured,
           CASE
               WHEN NOT matured THEN 'Immature'
               WHEN NOT fully_paid THEN 'Default, still short at extract'
               WHEN last_payment_after_dpd30 THEN 'Default, later fully repaid'
               ELSE 'Fully paid by DPD30'
           END AS outcome
    FROM loan_flags
)
SELECT COUNT(*) AS loans_disbursed,
       SUM(matured) AS mature_loans,
       SUM(outcome = 'Immature') AS immature_loans,
       SUM(outcome = 'Fully paid by DPD30') AS fully_paid_by_dpd30,
       SUM(outcome = 'Default, later fully repaid') AS default_later_repaid,
       SUM(outcome = 'Default, still short at extract') AS default_still_short,
       SUM(outcome LIKE 'Default,%') AS dpd30_defaults,
       ROUND(100.0 * SUM(outcome LIKE 'Default,%') / SUM(matured), 2) AS dpd30_rate_pct
FROM loan_outcomes;

Out[0]: 
+-----------------+--------------+----------------+---------------------+----------------------+---------------------+----------------+----------------+
| loans_disbursed | mature_loans | immature_loans | fully_paid_by_dpd30 | default_later_repaid | default_still_short | dpd30_defaults | dpd30_rate_pct |
+-----------------+--------------+----------------+---------------------+----------------------+---------------------+----------------+----------------+
|       1310      |     1040     |      270       |         964         |          0           |          76         |       76       |      7.31      |
+-----------------+--------------+----------------+---------------------+----------------------+---------------------+----------------+----------------+


Running query in 'sqlite://'

In [10]:
%%sql
SELECT COUNT(DISTINCT plus.user_id) AS unique_plus_subscribers_during_q3
FROM plus
CROSS JOIN report_dates
WHERE plus.plus_start_date < q3_end
  AND (plus.plus_end_date IS NULL OR plus.plus_end_date > q3_start);

Out[0]: 
+-----------------------------------+
| unique_plus_subscribers_during_q3 |
+-----------------------------------+
|                 93                |
+-----------------------------------+


Running query in 'sqlite://'